# GalactICS → ntropy: NFW halo end-to-end

*A walkthrough notebook for the PyGalactICS rewrite — from graduate-school Fortran to a testable Python library.*

## What is GalactICS?

**GalactICS** (Kuijken & Dubinski) is a code package for building **N-body initial conditions (ICs)** for idealized galaxy models — not a time integrator. Its job is to produce particle positions and velocities that are as close as practical to **dynamical equilibrium** in a chosen gravitational potential, so downstream simulations (Gadget, GIZMO, etc.) start from a physically meaningful state rather than an arbitrary point cloud.

At a high level GalactICS:

1. **Defines a galaxy model** — halo (often NFW), stellar disk(s), bulge, gas, each with parametric density laws.
2. **Solves for the self-consistent potential** — the `dbh` program iterates a multipole Poisson solver until the density implied by distribution functions (DFs) and analytic components matches the potential they generate (tidal radius convergence).
3. **Builds distribution functions** — halos and bulges use **Eddington inversion** on spherical ρ(r); disks use action-based DFs with asymmetric-drift corrections (`diskdf`).
4. **Samples particles** — `genhalo`, `gendisk`, `genbulge` draw phase-space points from those DFs and write ASCII particle files.

GalactICS assumes **G = 1** with length in kpc, velocity in 100 km/s, and a fixed mass unit (2.325×10⁹ M☉). That is the same unit system used throughout this repository.

## What this notebook does

This notebook is the **integrated test** the Python rewrite was built for:

| Stage | Tool | Purpose |
|-------|------|---------|
| IC generation | **galacticsics** + legacy `dbh` / `genhalo` | Real GalactICS pipeline via a typed Python API |
| Validation | **ntropy** | Short self-gravitating evolution to check IC quality |

We walk through a **spherical NFW halo** (simplest non-trivial case), then:

1. Sample halo particles through `dbh` → `genhalo`
2. Plot the initial **ρ(r)** to confirm the sampled profile looks like a halo
3. Benchmark **brute-force** vs **Barnes–Hut** gravity (force accuracy vs θ)
4. Evolve the IC briefly and plot **ρ(r)** again to see whether the profile is stable

Figures and particle files are written to `notebooks/artifacts/` (gitignored).

**Prerequisites:** `make install-dev` (builds `legacy/bin`, installs both packages), plus `pip install jupyter matplotlib`.

## Repository layout (after the rewrite)

The graduate-school Fortran/C still runs, but Python owns the API and tests:

```
GalaxyModel  →  dbh (solve_potential)  →  genhalo  →  ParticleSet  →  ntropy Simulation
     ↑              legacy/bin/              legacy/bin/              src/ntropy/
 galacticsics      subprocess bridge        subprocess bridge
```

- **`galacticsics`** (`src/galacticsics/`) — models, I/O, `solve_potential`, `sample_galaxy`, artifact generation.
- **`legacy/`** — original `dbh.f`, `genhalo`, etc. (gitignored; `make legacy-build`).
- **`ntropy`** (`src/ntropy/`) — minimal N-body integrator that reads the same ASCII particle format.

**Units** (shared everywhere):

| Quantity | Unit |
|----------|------|
| Length | kpc |
| Velocity | 100 km/s |
| Mass | 2.325×10⁹ M☉ |
| G | 1 |

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np

from ntropy.analysis.density import bin_spherical_density, compare_density_profiles
from ntropy.config import ForceConfig, IntegratorConfig, ParallelConfig, RunConfig
from ntropy.forces.brute import compute_forces_brute
from ntropy.forces.bhtree import compute_forces_bh
from ntropy.integrations.galacticsics import (
    galacticsics_available,
    nfw_halo_model_fast,
    sample_galacticsics_halo,
)
from ntropy.simulation import Simulation

if not galacticsics_available():
    raise RuntimeError(
        "galacticsics + legacy binaries required. From repo root: make install-dev"
    )

# Repo root (parent of notebooks/)
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
elif not (REPO / "src" / "ntropy").exists():
    REPO = REPO.parent  # running from notebooks/ subdir in some IDEs

ARTIFACTS = REPO / "notebooks" / "artifacts" / "nfw_walkthrough"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Keep matplotlib cache inside artifacts (useful in CI / sandboxed envs)
os.environ.setdefault("MPLCONFIGDIR", str(ARTIFACTS / ".matplotlib"))

import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

print(f"Artifacts → {ARTIFACTS}")

## 1. GalactICS: model → dbh → genhalo

### Pipeline steps

1. **`GalaxyModel`** — describes which components exist and the numerical grid (`nr`, `dr`, `lmax`).
2. **`solve_potential` (`dbh`)** — self-consistent multipole solve. For a pure halo with `lmax=0` this reduces to a spherical problem: build ρ(r) from the NFW law, invert the DF, tabulate `dfnfw.dat`, and write `dbh.dat` / `h.dat`.
3. **`genhalo`** — Monte Carlo sample of (r, v) from `dfnfw.dat`, output ASCII particles.

`sample_galacticsics_halo()` in `ntropy.integrations.galacticsics` wraps all three steps and converts the result to ntropy's `ParticleState`.

### `nfw_halo_model_fast()` — what it is and why

Notebooks and CI cannot wait for a full `nr=2000` production halo solve. The helper `nfw_halo_model_fast()` returns a **deliberately reduced** `GalaxyModel` that preserves the physics of an isolated NFW halo but finishes `dbh` in seconds:

| Parameter | Production-style (`nfw_halo_only`) | `nfw_halo_model_fast()` | Why reduced |
|-----------|-----------------------------------|-------------------------|-------------|
| `r_outer` (chalo) | 100 kpc | **60 kpc** | Smaller domain → fewer radial integrals |
| `a` (scale radius) | 10 kpc | **8 kpc** | Slightly more compact halo |
| `v0` | 2.0 (200 km/s) | **2.0** | Same potential depth scale |
| `dr_trunc` | 8–12 kpc | **8 kpc** | Outer truncation of NFW |
| `nr` (radial points) | 2000 | **800** | Coarser grid (like `reference_disk_halo`) |
| `lmax` | 0 | **0** | Purely spherical — no disk/bulge multipoles |

With `lmax=0` only the monopole (spherical) harmonic is active, which is correct for an isolated halo test. The model still runs the **real** `dbh` + Eddington DF machinery — it is not a toy analytic sampler.

**Softening** (`eps = 0.04` kpc below) is an ntropy parameter only; GalactICS particles are softened when we evolve them in ntropy, not during `genhalo`.

In [ ]:
model = nfw_halo_model_fast()
print(model)

GALACTICS_WORK = ARTIFACTS / "galacticsics_work"
EPS = 0.04

ic_result = sample_galacticsics_halo(
    model,
    n_particles=256,
    seed=-42,  # legacy ran3 convention (negative integer)
    work_dir=GALACTICS_WORK,
    eps=EPS,
    solve=True,
)

state = ic_result.state
particle_file = ARTIFACTS / "nfw_halo_galacticsics.dat"
state.write_ascii(particle_file)

print(f"N = {state.n}, total mass = {state.mass.sum():.2f}")
print(f"GalactICS work_dir = {ic_result.work_dir}")
print(f"dbh.dat present: {(ic_result.work_dir / 'dbh.dat').exists()}")
print(f"dfnfw.dat present: {(ic_result.work_dir / 'dfnfw.dat').exists()}")
print(f"Wrote {particle_file.name}")

## 2. Initial density plot — did `genhalo` sample the right halo?

### Why this plot?

GalactICS guarantees equilibrium only in the **smooth** DF limit. With finite N (here 256 particles) we expect Monte Carlo noise, but the **shape** of ρ(r) should still look like an NFW halo: roughly flat in the inner core, steepening toward the outer edge.

Binning particles into spherical shells and plotting ρ(r) answers:

- Did `dbh` + `genhalo` produce a spatial distribution consistent with an NFW profile?
- Are there obvious bugs in the Python bridge (wrong units, missing files, empty output)?

This is a **sanity check on the IC step**, before we spend time on dynamics.

### How we bin

Each shell receives the sum of particle masses between radii rᵢ and rᵢ₊₁; we divide by the shell volume 4π(rᵢ₊₁³ − rᵢ³)/3 to get a density estimate. We use log-log axes because NFW ρ(r) spans several decades.

In [ ]:
r_max = float(np.sqrt(np.sum(state.pos**2, axis=1)).max()) * 1.1
profile = bin_spherical_density(state.pos, state.mass, n_bins=20, r_max=r_max)

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(profile.r_mid, profile.rho, "o-", ms=5, alpha=0.8, label="genhalo particles")
ax.set_xlabel("r [kpc]")
ax.set_ylabel("ρ [M_unit / kpc³]")
ax.set_title("GalactICS NFW halo — initial ρ(r)")
ax.legend()
fig.tight_layout()
density_init_path = ARTIFACTS / "density_initial.png"
fig.savefig(density_init_path, dpi=150)
plt.show()
print(f"Saved {density_init_path.name}")

## 3. Force accuracy plots — can we trust the Barnes–Hut tree?

ntropy offers two force kernels. Before running a long simulation with the tree, we need to know **how wrong** it is compared to the exact pairwise sum.

| Method | Cost | Role in this notebook |
|--------|------|------------------------|
| **Brute force** | O(N²) | Reference truth at N = 256 |
| **Barnes–Hut** | O(N log N) | Production choice for larger N |

The tree opens a node when `size / distance ≥ θ` (larger θ → more aggressive grouping → faster but less accurate).

### Why these plots?

1. **Error vs θ (left panel)** — picks an operating point. We want the smallest θ that still gives acceptable error on this halo geometry. This is standard practice when calibrating tree codes for a new softening / particle layout.

2. **Error vs radius at fixed θ (right panel)** — reveals *where* the tree struggles. Inner regions with many close neighbours often show larger relative errors; the outskirts may be excellent. If errors were uniform in radius we might suspect a bug; radial structure is expected.

We report max, median, and RMS relative acceleration error ‖Δ**a**‖ / ‖**a**_brute‖ and save `force_accuracy.csv` for the blog post.

In [ ]:
pos, mass, eps = state.pos, state.mass, state.eps
acc_brute = compute_forces_brute(pos, mass, eps)
brute_norm = np.linalg.norm(acc_brute, axis=1)
brute_norm = np.maximum(brute_norm, 1e-30)

thetas = [0.2, 0.3, 0.4, 0.5, 0.7, 1.0]
rows = []

for theta in thetas:
    acc_bh = compute_forces_bh(pos, mass, eps, theta=theta)
    delta = acc_bh - acc_brute
    rel = np.linalg.norm(delta, axis=1) / brute_norm
    rows.append({
        "theta": theta,
        "max_rel": float(rel.max()),
        "median_rel": float(np.median(rel)),
        "rms_rel": float(np.sqrt(np.mean(rel**2))),
    })

print(f"{'theta':>6} {'max_rel':>10} {'median_rel':>12} {'rms_rel':>10}")
for row in rows:
    print(f"{row['theta']:6.1f} {row['max_rel']:10.4f} {row['median_rel']:12.4f} {row['rms_rel']:10.4f}")

csv_path = ARTIFACTS / "force_accuracy.csv"
with open(csv_path, "w") as f:
    f.write("theta,max_rel,median_rel,rms_rel\n")
    for row in rows:
        f.write(f"{row['theta']},{row['max_rel']},{row['median_rel']},{row['rms_rel']}\n")
print(f"\nSaved {csv_path.name}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

theta_arr = [r["theta"] for r in rows]
axes[0].semilogy(theta_arr, [r["max_rel"] for r in rows], "o-", label="max")
axes[0].semilogy(theta_arr, [r["median_rel"] for r in rows], "s-", label="median")
axes[0].set_xlabel("opening angle θ")
axes[0].set_ylabel("relative |Δa| / |a_brute|")
axes[0].set_title("BH force error vs θ")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Per-particle error at θ = 0.5
theta_demo = 0.5
acc_demo = compute_forces_bh(pos, mass, eps, theta=theta_demo)
rel_demo = np.linalg.norm(acc_demo - acc_brute, axis=1) / brute_norm
r = np.sqrt(np.sum(pos**2, axis=1))
axes[1].scatter(r, rel_demo, s=12, alpha=0.6, c=r, cmap="viridis")
axes[1].set_xlabel("r [kpc]")
axes[1].set_ylabel("relative force error")
axes[1].set_title(f"Per-particle error at θ = {theta_demo}")
fig.colorbar(axes[1].collections[0], ax=axes[1], label="r [kpc]")
fig.tight_layout()
force_plot_path = ARTIFACTS / "force_accuracy.png"
fig.savefig(force_plot_path, dpi=150)
plt.show()
print(f"Saved {force_plot_path.name}")

## 4. Stability plot — is the GalactICS IC in equilibrium?

### Why this plot?

This is the **core integrated test**: GalactICS claims the particles are drawn from an equilibrium DF in the halo potential. If we place those particles in ntropy and let them interact via **self-gravity** (with Plummer softening), a good IC should show:

- **Small drift** in ρ(r) over a few dynamical times
- No systematic steepening or flattening of the profile

A bad IC (wrong DF, unit mismatch, non-equilibrium sampling) will show rapid profile evolution — particles phase-mix incorrectly, the core contracts or expands, and shell densities change by order unity.

### Run settings

- **Brute force** — exact gravity at this N, isolates IC quality from tree approximation error.
- **25 steps × dt = 0.002** — short deliberately; we are not simulating a Hubble time, only checking that the IC does not explode immediately.
- **Same binning as §2** — direct before/after comparison on the same radial grid.

The printed `max_drift` is the maximum relative change in any shell with enough particles; values ≲ 0.3 are typical for small N with softening (see `test_galacticsics_integration.py`).

In [ ]:
cfg = RunConfig()
cfg.integrator = IntegratorConfig(dt=0.002, n_steps=25)
cfg.force = ForceConfig(method="brute")
cfg.parallel = ParallelConfig(enabled=False)
cfg.output.write_final = False
cfg.output.every = 0

result = Simulation(cfg, state=state.copy()).run()

init_prof = bin_spherical_density(
    result.initial_state.pos, result.initial_state.mass, n_bins=20, r_max=r_max
)
final_prof = bin_spherical_density(
    result.final_state.pos, result.final_state.mass, n_bins=20, r_max=r_max
)
max_drift = compare_density_profiles(init_prof, final_prof, min_count=3)
print(f"Max relative spherical density change after {cfg.integrator.n_steps} steps: {max_drift:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(init_prof.r_mid, init_prof.rho, "o-", label="t = 0 (GalactICS IC)")
ax.loglog(final_prof.r_mid, final_prof.rho, "s-", label=f"t = {cfg.integrator.n_steps} dt (ntropy)")
ax.set_xlabel("r [kpc]")
ax.set_ylabel("ρ [M_unit / kpc³]")
ax.set_title("Density stability — GalactICS IC + ntropy evolution")
ax.legend()
fig.tight_layout()
density_final_path = ARTIFACTS / "density_stability.png"
fig.savefig(density_final_path, dpi=150)
plt.show()
print(f"Saved {density_final_path.name}")

## 5. Blog series — where to go next

This notebook is intentionally small and reproducible. Future posts in the series might cover:

| Topic | Package | Notes |
|-------|---------|-------|
| Disk + halo from reference artifacts | `galacticsics` → `ntropy` | `sample_galacticsics_galaxy()` + `tests/generated/reference` |
| Halo-first two-step workflow | `galacticsics` | `examples/halo_first_workflow.py` |
| Exponential disk Σ(R) tests | `galacticsics` gendisk → `ntropy` | `test_galacticsics_integration.py` |
| MPI domain decomposition | `ntropy` | `mpirun -n 4 ntropy-run config.json` |
| Fitting NFW to particles | `galacticsics.fitting` | observational rotation curves |

**Cursor angle:** the rewrite isolates Fortran under `legacy/`, exposes typed Python APIs, replaces checked-in golden files with generated artifacts, and uses **integrated tests** (`ntropy/tests/test_galacticsics_integration.py`) to prove the old IC code still works in the new stack.

> **Note:** `ntropy.ics.*` provides pure-Python analytic samplers for fast unit tests without a legacy build. Production validation should use the **galacticsics** path shown here.

All artifacts from this run live under `notebooks/artifacts/nfw_walkthrough/` and are safe to delete and regenerate.